# Validation plots for pz data challenge (part 0)

#### Standard imports

In [ ]:
import numpy as np
import tables_io
import os
import matplotlib.pyplot as plt

#### Data setup and reading files

In [ ]:
# Modify this if you put the data somewhere else
basedir = "../public"

surveys = ["cardinal", "flagship"]
df = {}
# Modify this to check different files
name1 = "pz_challenge_taskset_1_"
name2 = "training_10yr"
savedir = f"{name1}{name2}"
print(savedir)
try:
    os.makedirs(savedir)
except:
    pass
for survey in surveys:
    infile = f"{name1}{survey}_{name2}.hdf5"
    tmpdata = tables_io.read(os.path.join(basedir,infile))
    df[survey] = tables_io.convert(tmpdata, tables_io.types.PD_DATAFRAME)
nsurv = len(surveys)

#### Load up the template curves

In [ ]:
# CWW colors to overlay
temps = ['El_B2004a','Sbc_B2004a', 'Scd_B2004a', 'Im_B2004a','ssp_25Myr_z008', 'ssp_5Myr_z008']
cols = ['r','g','m','b','dodgerblue','gray']
cfile = "CWWSB_LSST_Roman_colors_YJH.hdf5"

cdata = tables_io.read(cfile)

#### Simple check on data content

In [ ]:
df['flagship'].info()

In [ ]:
df['cardinal'].info()

#### Compute colors and add them to the dataframe

In [ ]:
# add colors to dataframe
bands = ['u','g','r','i','z','y','Y','J','H']
for survey in surveys:
    for i in range(8):
        if i<5:
            df[survey][f'{bands[i]}{bands[i+1]}'] = df[survey][f'mag_{bands[i]}_lsst'] - df[survey][f'mag_{bands[i+1]}_lsst']
    
        elif i==5:
            df[survey][f'{bands[i]}{bands[i+1]}'] = df[survey][f'mag_{bands[i]}_lsst'] - df[survey][f'mag_{bands[i+1]}_roman']
        else:
            df[survey][f'{bands[i]}{bands[i+1]}'] = df[survey][f'mag_{bands[i]}_roman'] - df[survey][f'mag_{bands[i+1]}_roman']

# Redshift histograms

In [ ]:
def plotredshifts(nzbins=100, logax=False):
    plt.figure(figsize=(10,8))
    for survey in surveys:    
       plt.hist(df[survey]['redshift'], bins=np.linspace(0,2.35, nzbins), alpha=0.5, label=survey)
    if logax:
        plt.yscale('log')
    plt.legend(loc='upper right', fontsize=12)
    plt.xlabel("redshift", fontsize=14)
    plt.ylabel("Number", fontsize=14)
    plt.xlim(0,3)
    if logax:
        outname = f"./{savedir}/Nz_hist_log.jpg"
    else:
        outname = f"./{savedir}/NZ_hist.jpg"
    plt.savefig(outname, format="jpg")

In [ ]:
plotredshifts(100, True)

In [ ]:
plotredshifts(100, False)

# color redshift and color color definitions

In [ ]:
def szcol(col, ylim=None):
    plt.figure(figsize=(8,6))
    for survey in surveys:
        plt.scatter(df[survey]['redshift'], df[survey][col], s=1, alpha=0.1, label=survey)
    for temp,colx in zip(temps, cols):
        plt.plot(cdata['redshift'][:300],cdata[f"{temp}_{col}"][:300], color=colx)
    plt.xlim(0,3)
    plt.ylim(ylim)
    plt.xlabel("redshift", fontsize=13)
    plt.ylabel(f"{col[0]} - {col[1]}", fontsize=13)
    leg = plt.legend(loc='upper right', fontsize=12)
    for lh in leg.legend_handles:
        lh.set_alpha(1)
    plt.savefig(f"./{savedir}/{col}_vs_sz.jpg", format="jpg")


def szcol2(col, ylim=None):
    fig, axs = plt.subplots(nsurv, 1, figsize=(12,13))
    surveyymins = []
    surveyymaxs = []
    if ylim==None:
        for i, survey in enumerate(surveys):
            surveyymins.append(np.amin(df[survey][col]))
            surveyymaxs.append(np.amax(df[survey][col]))
        ylim = (np.amin(np.array(surveyymins)), np.amax(np.array(surveyymaxs)))
    for i, survey in enumerate(surveys):
        axs[i].scatter(df[survey]['redshift'], df[survey][col], s=1, c='k', alpha=0.5, label=survey)
        for temp, colx in zip(temps, cols):
            axs[i].plot(cdata['redshift'][:300], cdata[f"{temp}_{col}"][:300], color=colx)
        axs[i].set_xlim(0,3)
        axs[i].set_ylim(ylim)
        axs[i].set_xlabel("redshift", fontsize=12)
        axs[i].set_ylabel(f"{col[0]} - {col[1]}", fontsize=12)
        leg = axs[i].legend(loc='upper right', fontsize=12)
        for lh in leg.legend_handles:
            lh.set_alpha(1)
    plt.savefig(f"./{savedir}/{col}_vs_sz_sidebyside.jpg", format="jpg")

def colcol(col1, col2, xlim=None, ylim=None):
    plt.figure(figsize=(9,9))
    for survey in surveys:
        plt.scatter(df[survey][col1], df[survey][col2], s=1, alpha=0.1, label=survey)
        for temp,colx in zip(temps, cols):
            plt.plot(cdata[f"{temp}_{col1}"][:300],cdata[f"{temp}_{col2}"][:300], color=colx)
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.xlabel(f"{col1[0]} - {col1[1]}", fontsize=13)
    plt.ylabel(f"{col2[0]} - {col2[1]}", fontsize=13)
    leg = plt.legend(loc='upper left', fontsize=12)
    for lh in leg.legend_handles:
        lh.set_alpha(1)
    plt.savefig(f"./{savedir}/{col2}_vs_{col1}.jpg", format="jpg")
        
def colcol2(col1, col2, xlim=None, ylim=None):
    fig, axs = plt.subplots(1, 2, figsize=(13,6))
    # need limits first:
    surveyxmins = []
    surveyxmaxs = []
    surveyymins = []
    surveyymaxs = []
    if xlim==None and ylim==None:
        for i, survey in enumerate(surveys):
            surveyxmins.append(np.amin(df[survey][col1]))
            surveyxmaxs.append(np.amax(df[survey][col1]))
            surveyymins.append(np.amin(df[survey][col2]))
            surveyymaxs.append(np.amax(df[survey][col2]))
        xlim = (np.amin(np.array(surveyxmins)), np.amax(np.array(surveyxmaxs)))
        ylim = (np.amin(np.array(surveyymins)), np.amax(np.array(surveyymaxs)))
    for i, survey in enumerate(surveys):
        axs[i].scatter(df[survey][col1], df[survey][col2], s=1, c='k',alpha=0.3, label=survey)
        for temp,colx in zip(temps, cols):
            axs[i].plot(cdata[f"{temp}_{col1}"][:300],cdata[f"{temp}_{col2}"][:300], color=colx)
        if xlim==None and ylim==None:
            xlimmin = np.amin(df[survey][col1])
            xlimmax = np.amax(df[survey][col1])
            ylimmin = np.amin(df[survey][col2])
            ylimmax = np.amax(df[survey][col2])
            xlim = (xlimmin, xlimmax)
            ylim = (ylimmin, ylimmax)
        axs[i].set_xlim(xlim)
        axs[i].set_ylim(ylim)
        axs[i].set_xlabel(f"{col1[0]} - {col1[1]}", fontsize=12)
        axs[i].set_ylabel(f"{col2[0]} - {col2[1]}", fontsize=12)
        leg = axs[i].legend(loc='upper left', fontsize=12)
        for lh in leg.legend_handles:
            lh.set_alpha(1)
    plt.savefig(f"./{savedir}/{col2}_vs_{col1}_sidebyside.jpg", format="jpg")


In [ ]:
szcol2("gr", (-.2, 3.5))

In [ ]:
colxs = ['ug','gr','ri','iz','zy','yY','YJ','JH']
for i in range(7):
    if i==0:
        colcol2(colxs[i], colxs[i+1], (-2,5), (-1,3.5))
    else:
        colcol2(colxs[i], colxs[i+1])

In [ ]:
colxs = ['ug','gr','ri','iz','zy','yY','YJ','JH']
for i in range(7):
    if i==0:
        colcol(colxs[i], colxs[i+1], (-2,5), (-1,3.5))
    else:
        colcol(colxs[i], colxs[i+1])

In [ ]:
colxs = ['ug','gr','ri','iz','zy','yY','YJ','JH']
for i in range(8):
    if i==0:
        szcol2(colxs[i],(-2,5))
    else:
        szcol2(colxs[i])

In [ ]:
colxs = ['ug','gr','ri','iz','zy','yY','YJ','JH']
for i in range(8):
    if i==0:
        szcol(colxs[i],(-2,5))
    else:
        szcol(colxs[i])

# magnitude number counts plots

In [ ]:
bands = ['u','g','r','i','z','y']
nirbands = ['Y','J','H']
magcs = ['b','g','r','m','k','gray']
nirmagcs = ['r','purple', 'orange']

In [ ]:
def plotnumbercounts_lsst(xbins=np.linspace(15,28,101)):
    fig, axs = plt.subplots(6, nsurv, figsize=(6*nsurv, 30))
    for i, survey in enumerate(surveys):
        for j, (band, mc) in enumerate(zip(bands, magcs)):
            if j==0:
                bins = np.linspace(15, 35, 101)
            else:
                bins = xbins
            axs[j,i].hist(df[survey][f"mag_{band}_lsst"], bins=bins, color=mc, alpha=0.75, label=f"{survey} {band}")
            axs[j,i].set_yscale('log')
            axs[j,i].set_xlabel("magnitude", fontsize=14)
            axs[j,i].set_ylabel("Number", fontsize=14)
            axs[j,i].legend(loc='upper left', fontsize=12);
    plt.savefig(f"./{savedir}/mag_number_counts_Rubin.jpg", format="jpg")


def plotnumbercounts_roman(bins=np.linspace(15,25.5,101)):
    fig, axs = plt.subplots(3, nsurv, figsize=(6*nsurv, 15))
    for i, survey in enumerate(surveys):
        for j, (band, mc) in enumerate(zip(nirbands, nirmagcs)):
            axs[j,i].hist(df[survey][f"mag_{band}_roman"], bins=bins, color=mc, alpha=0.75, label=f"{survey} {band}")
            axs[j,i].set_yscale('log')
            axs[j,i].set_xlabel("magnitude", fontsize=14)
            axs[j,i].set_ylabel("Number", fontsize=14)
            axs[j,i].legend(loc='upper left', fontsize=12);
    plt.savefig(f"./{savedir}/mag_number_counts_Roman.jpg", format="jpg")


In [ ]:
plotnumbercounts_lsst()

In [ ]:
plotnumbercounts_roman()

# footprint

In [ ]:
def plotradec():
    fig, axs = plt.subplots(1,nsurv, figsize=(6*nsurv,6))
    for i, survey in enumerate(surveys):
        axs[i].scatter(df[survey]['ra'], df[survey]['dec'], s=.1, c='k', label=survey)
        axs[i].legend(loc='lower right', fontsize=12)
        axs[i].set_xlabel("RA", fontsize=14)
        axs[i].set_xlabel("DEC", fontsize=14)
    plt.savefig(f"./{savedir}/RA_DEC_footprint.jpg", format="jpg")

In [ ]:
plotradec()